# Polymarket Extreme Probability Visual Check

This notebook is the main exploratory visualization pass for the long-shot and favorite-bias thesis using the local `data/raw/data/polymarket` archive.

Core questions:
- Do low probability YES contracts quoted below `10%` resolve YES less often than the market implied?
- Do high probability YES contracts quoted above `90%` resolve YES more often than the market implied?
- Does the directional pattern survive the primary `threshold-entry` sampling rule instead of relying on raw all-tick counts alone?

The notebook rebuilds the canonical DuckDB warehouse and the Milestone 3 analysis tables if they are missing, then uses a small set of clear visualization layers:
- archive and sample coverage
- a calibration gap scorecard with Wilson intervals and bootstrap gap intervals
- bucketed quoted-versus-realized calibration plots
- a time-to-expiry segmentation heatmap
- a probability distribution view so the thesis buckets are easy to contextualize


In [ ]:
import sys
from pathlib import Path


_REPO_SEARCH_ROOT_NAMES = ("Desktop", "Documents", "code", "personal")
_REPO_SEARCH_SKIP_DIRS = {".git", ".venv", "__pycache__", "node_modules"}


def is_repo_root(candidate: Path) -> bool:
    return (candidate / "PROJECT_SPEC.md").exists() and (candidate / "src").is_dir()


def iter_search_roots(start: Path) -> tuple[Path, ...]:
    roots = []
    seen = set()

    def add(candidate: Path) -> None:
        resolved = candidate.resolve()
        key = resolved.as_posix()
        if key in seen or not resolved.exists() or not resolved.is_dir():
            return
        seen.add(key)
        roots.append(resolved)

    add(start)
    for parent in start.parents:
        add(parent)

    home = Path.home().resolve()
    add(home)
    for root_name in _REPO_SEARCH_ROOT_NAMES:
        add(home / root_name)
        add(home / "Desktop" / root_name)

    return tuple(roots)


def iter_descendants(root: Path, max_depth: int = 4):
    stack = [(root, 0)]
    seen = set()
    while stack:
        current, depth = stack.pop()
        key = current.as_posix()
        if key in seen:
            continue
        seen.add(key)
        yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in reversed(children):
            if not child.is_dir() or child.name in _REPO_SEARCH_SKIP_DIRS:
                continue
            stack.append((child, depth + 1))


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if is_repo_root(candidate):
            return candidate

    for search_root in iter_search_roots(start):
        for candidate in iter_descendants(search_root):
            if is_repo_root(candidate):
                return candidate

    raise RuntimeError("Could not locate the repository root from the notebook working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


if "REPO_ROOT" not in globals():
    def find_repo_root(start: Path) -> Path:
        for candidate in [start, *start.parents]:
            if (candidate / "PROJECT_SPEC.md").exists() and (candidate / "src").is_dir():
                return candidate

        fallback = Path("/Users/marinmestrovic/Desktop/personal/polymarket-extreme-probability")
        if (fallback / "PROJECT_SPEC.md").exists() and (fallback / "src").is_dir():
            return fallback

        raise RuntimeError("Could not locate the repository root from the notebook working directory.")

    REPO_ROOT = find_repo_root(Path.cwd().resolve())

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter

from src.research import ensure_notebook_study_context

plt.style.use("bmh")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.facecolor": "#f5efe2",
        "axes.facecolor": "#fffaf2",
        "axes.edgecolor": "#3f3a2f",
        "axes.labelcolor": "#2f2a21",
        "axes.titleweight": "bold",
        "axes.titlesize": 14,
        "xtick.color": "#2f2a21",
        "ytick.color": "#2f2a21",
        "grid.color": "#d8cfbf",
        "grid.alpha": 0.35,
        "text.color": "#1f1a14",
    }
)
pd.options.display.float_format = lambda value: f"{value:,.4f}"

SAMPLING_VIEW_ORDER = ["all_ticks", "threshold_entry"]
SAMPLING_VIEW_LABELS = {
    "all_ticks": "All ticks",
    "threshold_entry": "Threshold entry",
}
PROBABILITY_BUCKET_ORDER = ["low_probability", "high_probability"]
PROBABILITY_BUCKET_LABELS = {
    "low_probability": "<10% YES",
    "high_probability": ">90% YES",
}
TIME_TO_EXPIRY_ORDER = [
    "under_24h",
    "one_to_seven_days",
    "over_seven_days",
    "unknown",
]
TIME_TO_EXPIRY_LABELS = {
    "under_24h": "<24h",
    "one_to_seven_days": "1-7d",
    "over_seven_days": ">7d",
    "unknown": "Unknown",
}
PALETTE = {
    "quoted": "#355070",
    "realized": "#2a9d8f",
    "low_probability": "#f4a261",
    "high_probability": "#264653",
}
VIEW_COLORS = {
    "all_ticks": "#9c6644",
    "threshold_entry": "#355070",
}


def format_percent(value: float | object) -> str:
    if pd.isna(value):
        return "n/a"
    return f"{value:.1%}"


In [ ]:
FORCE_REBUILD = False
BOOTSTRAP_SAMPLES = 400
RANDOM_SEED = 17

context = ensure_notebook_study_context(
    start_dir=REPO_ROOT,
    force_rebuild=FORCE_REBUILD,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    random_seed=RANDOM_SEED,
)
warehouse_path = context.paths.warehouse_path
raw_dir = context.paths.raw_dir
polymarket_root = context.paths.polymarket_root

connection = duckdb.connect(str(warehouse_path), read_only=True)

status_lines = [
    f"- Raw archive root: `{raw_dir.as_posix()}`",
    f"- Polymarket source: `{polymarket_root.as_posix()}`",
    f"- DuckDB warehouse: `{warehouse_path.as_posix()}`",
    f"- Canonical tables rebuilt in this run: `{context.canonical_built}`",
    f"- Analysis tables rebuilt in this run: `{context.analysis_built}`",
]
if context.analysis_result is not None and context.analysis_result.missing_expected_venues:
    missing_venues = ", ".join(context.analysis_result.missing_expected_venues)
    status_lines.append(f"- Missing expected venues in the current warehouse: `{missing_venues}`")

display(Markdown("## Local Study Context\n" + "\n".join(status_lines)))

## Coverage and Cohort Shape

Start by confirming what this notebook is actually reading. For the current local setup, the thesis visuals should be driven by the Polymarket `markets` snapshot files, with `trades` and `legacy_trades` still deferred. The tables below show archive coverage plus the size of the resolved market cohort that flows into the calibration analysis.

In [ ]:
inventory_df = connection.execute(
    """
    SELECT
        dataset,
        COUNT(*) AS file_count,
        ROUND(SUM(size_bytes) / 1024.0 / 1024.0, 2) AS total_size_mib
    FROM archive_inventory
    GROUP BY 1
    ORDER BY CASE dataset
        WHEN 'markets' THEN 0
        WHEN 'trades' THEN 1
        WHEN 'legacy_trades' THEN 2
        WHEN 'blocks' THEN 3
        ELSE 4
    END
    """
).df()

coverage_df = connection.execute(
    """
    WITH market_coverage AS (
        SELECT
            venue,
            COUNT(*) AS market_count,
            SUM(tick_observation_count) AS tick_observation_count,
            MIN(first_observation_time_utc) AS first_observation_time_utc,
            MAX(last_observation_time_utc) AS last_observation_time_utc
        FROM market_catalog
        GROUP BY 1
    ),
    threshold_coverage AS (
        SELECT
            venue,
            COUNT(*) FILTER (WHERE contract_side = 'YES') AS threshold_entry_event_count,
            COUNT(DISTINCT market_id) FILTER (WHERE contract_side = 'YES') AS threshold_market_count
        FROM threshold_entry_events
        GROUP BY 1
    ),
    extreme_coverage AS (
        SELECT
            venue,
            COUNT(*) AS extreme_tick_count,
            COUNT(DISTINCT market_id) AS extreme_tick_market_count
        FROM tick_observations
        WHERE contract_side = 'YES'
          AND (probability < 0.10 OR probability > 0.90)
        GROUP BY 1
    ),
    resolution_coverage AS (
        SELECT
            venue,
            COUNT(*) FILTER (WHERE contract_side = 'YES' AND resolved_outcome = 'YES') AS yes_resolution_count,
            COUNT(*) FILTER (WHERE contract_side = 'YES' AND resolved_outcome = 'NO') AS no_resolution_count
        FROM resolution_outcomes
        GROUP BY 1
    )
    SELECT
        market_coverage.venue,
        market_coverage.market_count,
        market_coverage.tick_observation_count,
        extreme_coverage.extreme_tick_count,
        extreme_coverage.extreme_tick_market_count,
        threshold_coverage.threshold_entry_event_count,
        threshold_coverage.threshold_market_count,
        resolution_coverage.yes_resolution_count,
        resolution_coverage.no_resolution_count,
        market_coverage.first_observation_time_utc,
        market_coverage.last_observation_time_utc
    FROM market_coverage
    LEFT JOIN threshold_coverage USING (venue)
    LEFT JOIN extreme_coverage USING (venue)
    LEFT JOIN resolution_coverage USING (venue)
    ORDER BY venue
    """
).df()

display(inventory_df)
display(coverage_df)

## Thesis Scorecard

This is the fastest visual read on whether the thesis holds. The table keeps both the quoted probability and realized YES rate side by side, plus the calibration gap (`empirical_yes_rate - average_quoted_probability`).

Interpretation:
- Negative gap in the low-probability bucket supports long-shot overvaluation.
- Positive gap in the high-probability bucket supports favorite underpricing.
- The Wilson interval columns belong to the realized YES rate; the plot below uses market-clustered bootstrap intervals around the calibration gap because that is the quantity we care about most here.

In [ ]:
summary_df = connection.execute(
    """
    SELECT
        sampling_view,
        venue,
        probability_bucket,
        observation_count,
        market_count,
        contract_count,
        average_quoted_probability,
        empirical_yes_rate,
        calibration_gap,
        wilson_interval_lower,
        wilson_interval_upper,
        bootstrap_gap_lower,
        bootstrap_gap_upper,
        sample_caveat
    FROM calibration_summaries
    WHERE venue = 'polymarket'
    ORDER BY
        CASE sampling_view WHEN 'all_ticks' THEN 0 ELSE 1 END,
        CASE probability_bucket WHEN 'low_probability' THEN 0 ELSE 1 END
    """
).df()

display_df = summary_df.copy()
display_df["sampling_view"] = display_df["sampling_view"].map(SAMPLING_VIEW_LABELS)
display_df["probability_bucket"] = display_df["probability_bucket"].map(PROBABILITY_BUCKET_LABELS)
display_df["quoted_probability"] = display_df["average_quoted_probability"].map(format_percent)
display_df["realized_yes_rate"] = display_df["empirical_yes_rate"].map(format_percent)
display_df["calibration_gap"] = display_df["calibration_gap"].map(format_percent)
display_df["Wilson yes-rate interval"] = display_df.apply(
    lambda row: f"{format_percent(row['wilson_interval_lower'])} to {format_percent(row['wilson_interval_upper'])}",
    axis=1,
)
display_df["Bootstrap gap interval"] = display_df.apply(
    lambda row: (
        f"{format_percent(row['bootstrap_gap_lower'])} to {format_percent(row['bootstrap_gap_upper'])}"
        if not pd.isna(row['bootstrap_gap_lower'])
        else "n/a"
    ),
    axis=1,
)

display(
    display_df[
        [
            "sampling_view",
            "probability_bucket",
            "observation_count",
            "market_count",
            "quoted_probability",
            "realized_yes_rate",
            "calibration_gap",
            "Wilson yes-rate interval",
            "Bootstrap gap interval",
            "sample_caveat",
        ]
    ]
)

plot_df = summary_df.copy()
plot_df["gap_pct"] = plot_df["calibration_gap"] * 100.0
plot_df["gap_low_pct"] = plot_df["bootstrap_gap_lower"].fillna(plot_df["calibration_gap"]) * 100.0
plot_df["gap_high_pct"] = plot_df["bootstrap_gap_upper"].fillna(plot_df["calibration_gap"]) * 100.0

x = np.arange(len(PROBABILITY_BUCKET_ORDER))
width = 0.36
fig, ax = plt.subplots(figsize=(10, 5))

for index, sampling_view in enumerate(SAMPLING_VIEW_ORDER):
    subset = (
        plot_df[plot_df["sampling_view"] == sampling_view]
        .set_index("probability_bucket")
        .reindex(PROBABILITY_BUCKET_ORDER)
        .reset_index()
    )
    centers = x + (index - 0.5) * width
    y = subset["gap_pct"].to_numpy()
    lower = y - subset["gap_low_pct"].to_numpy()
    upper = subset["gap_high_pct"].to_numpy() - y

    ax.bar(
        centers,
        y,
        width=width,
        label=SAMPLING_VIEW_LABELS[sampling_view],
        color=VIEW_COLORS[sampling_view],
        edgecolor="#2f2a21",
        alpha=0.88,
    )
    ax.errorbar(
        centers,
        y,
        yerr=[lower, upper],
        fmt="none",
        ecolor="#2f2a21",
        capsize=4,
        lw=1.2,
    )

    for center, value, obs in zip(centers, y, subset["observation_count"]):
        vertical_alignment = "bottom" if value >= 0 else "top"
        offset = 1.8 if value >= 0 else -2.8
        ax.text(center, value + offset, f"n={int(obs)}", ha="center", va=vertical_alignment, fontsize=9)

ax.axhline(0, color="#2f2a21", lw=1.0)
ax.set_xticks(x)
ax.set_xticklabels([PROBABILITY_BUCKET_LABELS[bucket] for bucket in PROBABILITY_BUCKET_ORDER])
ax.set_ylabel("Calibration gap")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_title("Polymarket extreme-bucket calibration gap")
ax.legend(title="Sampling view")
plt.show()

## Bucketed Calibration Plot

This scatter is the cleanest visual answer to the project thesis. Each point is an extreme bucket. Perfect calibration sits on the dashed diagonal. For the thesis to hold visually:
- low probability buckets should fall **below** the diagonal because realized YES rates are lower than quoted
- high probability buckets should move **above** the diagonal because realized YES rates are higher than quoted

Point size scales with the number of observations contributing to that bucket, so a dramatic but tiny bucket does not read the same way as a broad cohort.

In [ ]:
bucket_df = connection.execute(
    """
    WITH observations AS (
        SELECT
            'all_ticks' AS sampling_view,
            tick.venue,
            tick.market_id,
            CAST(tick.probability AS DOUBLE) AS probability,
            CASE resolution.resolved_outcome WHEN 'YES' THEN 1.0 ELSE 0.0 END AS resolved_yes
        FROM tick_observations AS tick
        INNER JOIN resolution_outcomes AS resolution
            ON resolution.venue = tick.venue
           AND resolution.market_id = tick.market_id
           AND resolution.contract_id = tick.contract_id
           AND resolution.contract_side = tick.contract_side
        WHERE tick.contract_side = 'YES'

        UNION ALL

        SELECT
            'threshold_entry' AS sampling_view,
            event.venue,
            event.market_id,
            CAST(event.probability AS DOUBLE) AS probability,
            CASE event.resolved_outcome WHEN 'YES' THEN 1.0 ELSE 0.0 END AS resolved_yes
        FROM threshold_entry_events AS event
        WHERE event.contract_side = 'YES'
    ),
    extreme_observations AS (
        SELECT
            sampling_view,
            venue,
            market_id,
            probability,
            resolved_yes,
            CASE
                WHEN probability < 0.10
                    THEN LEAST(CAST(FLOOR(probability / 0.02) AS INTEGER), 4)
                WHEN probability > 0.90
                    THEN LEAST(CAST(FLOOR((probability - 0.90) / 0.02) AS INTEGER), 4) + 45
                ELSE NULL
            END AS bucket_index
        FROM observations
        WHERE probability < 0.10 OR probability > 0.90
    )
    SELECT
        sampling_view,
        venue,
        CASE
            WHEN bucket_index < 45 THEN bucket_index * 0.02
            ELSE 0.90 + (bucket_index - 45) * 0.02
        END AS bucket_start,
        CASE
            WHEN bucket_index < 45 THEN LEAST(bucket_index * 0.02 + 0.02, 0.10)
            ELSE LEAST(0.90 + (bucket_index - 45) * 0.02 + 0.02, 1.00)
        END AS bucket_end,
        COUNT(*) AS observation_count,
        COUNT(DISTINCT market_id) AS market_count,
        AVG(probability) AS average_quoted_probability,
        AVG(resolved_yes) AS empirical_yes_rate,
        AVG(resolved_yes) - AVG(probability) AS calibration_gap
    FROM extreme_observations
    WHERE venue = 'polymarket' AND bucket_index IS NOT NULL
    GROUP BY 1, 2, 3, 4
    ORDER BY CASE sampling_view WHEN 'all_ticks' THEN 0 ELSE 1 END, bucket_start
    """
).df()

bucket_df["bucket_label"] = bucket_df.apply(
    lambda row: f"{row['bucket_start']:.0%}-{row['bucket_end']:.0%}",
    axis=1,
)
bucket_df["probability_bucket"] = np.where(
    bucket_df["bucket_start"] < 0.10,
    "low_probability",
    "high_probability",
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)

for ax, sampling_view in zip(axes, SAMPLING_VIEW_ORDER):
    subset = (
        bucket_df[bucket_df["sampling_view"] == sampling_view]
        .sort_values("average_quoted_probability")
        .reset_index(drop=True)
    )

    ax.plot([0, 1], [0, 1], linestyle="--", color="#8d8a80", lw=1.5, label="Perfect calibration")
    ax.scatter(
        subset["average_quoted_probability"],
        subset["empirical_yes_rate"],
        s=60 + subset["observation_count"].clip(lower=1) * 1.2,
        c=subset["probability_bucket"].map(PALETTE),
        alpha=0.92,
        edgecolor="#2f2a21",
    )

    for row in subset.itertuples():
        ax.annotate(
            f"{row.bucket_label}\nn={row.observation_count}",
            (row.average_quoted_probability, row.empirical_yes_rate),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=8,
        )

    ax.set_title(SAMPLING_VIEW_LABELS[sampling_view])
    ax.set_xlabel("Average quoted YES probability")
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

axes[0].set_ylabel("Empirical YES resolution rate")
fig.suptitle(
    "Bucketed quoted-vs-realized calibration\n"
    "Points below the diagonal mean YES contracts were overpriced",
    y=1.02,
)
plt.show()

## Threshold-Entry Time-to-Expiry Segmentation

The main inference path in this repository is the `threshold-entry` view, not the raw all-tick summary. This heatmap checks whether the sign and magnitude of the calibration gap are being driven by one narrow expiry regime.

A robust thesis should not disappear entirely once the data is split by time-to-expiry. At the same time, the annotations keep the sample size visible so small cells do not get over-interpreted.

In [ ]:
segment_df = connection.execute(
    """
    SELECT
        probability_bucket,
        segment_value,
        observation_count,
        market_count,
        average_quoted_probability,
        empirical_yes_rate,
        calibration_gap,
        bootstrap_gap_lower,
        bootstrap_gap_upper,
        sample_caveat
    FROM calibration_segments
    WHERE venue = 'polymarket'
      AND sampling_view = 'threshold_entry'
      AND segment_name = 'time_to_expiry'
    ORDER BY
        CASE probability_bucket WHEN 'low_probability' THEN 0 ELSE 1 END,
        CASE segment_value
            WHEN 'under_24h' THEN 0
            WHEN 'one_to_seven_days' THEN 1
            WHEN 'over_seven_days' THEN 2
            ELSE 3
        END
    """
).df()

heatmap_df = (
    segment_df.pivot(index="probability_bucket", columns="segment_value", values="calibration_gap")
    .reindex(index=PROBABILITY_BUCKET_ORDER, columns=TIME_TO_EXPIRY_ORDER)
)
observation_df = (
    segment_df.pivot(index="probability_bucket", columns="segment_value", values="observation_count")
    .reindex(index=PROBABILITY_BUCKET_ORDER, columns=TIME_TO_EXPIRY_ORDER)
)

heatmap_values = heatmap_df.to_numpy(dtype=float)
finite_values = heatmap_values[np.isfinite(heatmap_values)]
scale = float(np.max(np.abs(finite_values))) if finite_values.size else 0.05
scale = max(scale, 0.05)

fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.imshow(heatmap_values * 100.0, cmap="RdBu_r", vmin=-scale * 100.0, vmax=scale * 100.0)

for row_index, bucket in enumerate(PROBABILITY_BUCKET_ORDER):
    for column_index, segment in enumerate(TIME_TO_EXPIRY_ORDER):
        value = heatmap_df.loc[bucket, segment]
        observations = observation_df.loc[bucket, segment]
        label = "n/a" if pd.isna(value) else f"{value:.1%}\nn={int(observations)}"
        ax.text(column_index, row_index, label, ha="center", va="center", fontsize=9, color="#1f1a14")

ax.set_xticks(range(len(TIME_TO_EXPIRY_ORDER)))
ax.set_xticklabels([TIME_TO_EXPIRY_LABELS[value] for value in TIME_TO_EXPIRY_ORDER])
ax.set_yticks(range(len(PROBABILITY_BUCKET_ORDER)))
ax.set_yticklabels([PROBABILITY_BUCKET_LABELS[value] for value in PROBABILITY_BUCKET_ORDER])
ax.set_title("Threshold-entry calibration gap by time-to-expiry")
ax.set_xlabel("Time-to-expiry segment")
ax.set_ylabel("Extreme bucket")

colorbar = fig.colorbar(image, ax=ax, format=PercentFormatter(xmax=100))
colorbar.set_label("Calibration gap")
plt.show()

## Probability Distribution Context

This last view is not a calibration test by itself. It shows where the notebook is sampling from so the extreme buckets are easy to interpret in context. The shaded regions mark the thesis buckets used throughout the project.

In [ ]:
probability_distribution_df = connection.execute(
    """
    SELECT
        'all_ticks' AS sampling_view,
        CAST(probability AS DOUBLE) AS probability
    FROM tick_observations
    WHERE venue = 'polymarket' AND contract_side = 'YES'

    UNION ALL

    SELECT
        'threshold_entry' AS sampling_view,
        CAST(probability AS DOUBLE) AS probability
    FROM threshold_entry_events
    WHERE venue = 'polymarket' AND contract_side = 'YES'
    """
).df()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)

for ax, sampling_view in zip(axes, SAMPLING_VIEW_ORDER):
    subset = probability_distribution_df[probability_distribution_df["sampling_view"] == sampling_view]
    ax.hist(
        subset["probability"],
        bins=np.linspace(0.0, 1.0, 26),
        color=VIEW_COLORS[sampling_view],
        alpha=0.90,
        edgecolor="#fffaf2",
    )
    ax.axvspan(0.00, 0.10, color=PALETTE["low_probability"], alpha=0.18)
    ax.axvspan(0.90, 1.00, color=PALETTE["high_probability"], alpha=0.18)
    ax.set_title(SAMPLING_VIEW_LABELS[sampling_view])
    ax.set_xlabel("YES probability")
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

axes[0].set_ylabel("Observation count")
fig.suptitle("Where the analysis is sampling from", y=1.03)
plt.show()

## Reading the Thesis Quickly

Use the `threshold-entry` visuals as the primary signal.

A visually supportive result would look like this:
- the `<10% YES` bucket stays below the diagonal and below zero on the calibration-gap chart
- the `>90% YES` bucket moves above the diagonal and above zero
- the time-to-expiry heatmap keeps the same sign in more than one segment rather than collapsing into one tiny cell

A visually inconclusive result looks like this:
- the threshold-entry bars sit near zero or the bootstrap gap intervals still straddle zero
- the bucketed scatter does not separate clearly from the diagonal
- segmentation flips sign immediately once the sample is split

When that happens, treat the exploratory visualization as a stop signal unless a better price source or broader venue coverage changes the picture.